# From relational to graph databases

In [ ]:
# Uncomment to install packages
# !pip install -r requirements.txt

## MySQL settings

In [ ]:
PASSWORD = '<your_MYSQL_instance_password>'
import mysql.connector

connection = mysql.connector.connect(
  host="localhost",
  user="root",
  passwd=PASSWORD,
  database="steam_data")

In [ ]:
# Uses the MySQL function we created in the previous step
from graphtastic.database.mysql import query_mysql

## Schema design

### Ingestion considerations

In [ ]:
play_query = 'SELECT id, game_name, hours FROM steam_play'
play_data = query_mysql(play_query, password=PASSWORD)
print(play_data[:10])


In [ ]:
purchase_query = 'SELECT id, game_name FROM steam_purchase'
purchase_data = query_mysql(purchase_query,password=PASSWORD )
print(purchase_data[:10])

In [ ]:
users = set([row[0] for row in play_data] + [row[0] for row in purchase_data])
user_ids = {user_id: igraph_id for igraph_id, user_id in enumerate(users)}
print(len(user_ids))

In [ ]:
games = set([row[1] for row in play_data] + [row[1] for row in purchase_data])
game_ids = {user_id: igraph_id for igraph_id, user_id in enumerate(games, len(user_ids))}
print(len(game_ids))

In [ ]:
print(sorted(user_ids.values(), reverse=True)[:10])
print(sorted(game_ids.values(), reverse=False)[:10])

In [ ]:
all_ids = sorted(list(user_ids.values()) + list(game_ids.values()))
assert all_ids == list(range(len(all_ids)))

In [ ]:
import igraph as ig
g = ig.Graph(directed=True)

In [ ]:
user_ids = dict(sorted(user_ids.items(), key=lambda item: item[1]))
game_ids = dict(sorted(game_ids.items(), key=lambda item: item[1]))

In [ ]:
steam_user_ids = list(user_ids.keys())
steam_game_ids = list(game_ids.keys())

In [ ]:
g.add_vertices(len(steam_user_ids) + len(steam_game_ids))
assert len(g.vs) == len(steam_user_ids) + len(steam_game_ids)

In [ ]:
all_steam_ids = steam_user_ids + steam_game_ids

In [ ]:
node_types = ['user' for _ in steam_user_ids] + ['game' for _ in steam_game_ids]

In [ ]:
g.vs['steam_id'] = all_steam_ids
g.vs['type'] = node_types

In [ ]:
print(g.vs['steam_id'][:10])
print(g.vs['type'][:10])

In [ ]:
game_nodes = g.vs.select(type_eq='game')
print(len(game_nodes))

In [ ]:
purchase_edges = [[user_ids[user], game_ids[purchase]]
               	for user, purchase in purchase_data]
play_edges = [[user_ids[user], game_ids[game], hours]
           	for user, game, hours in play_data]

In [ ]:
g.add_edges([(n, m) for n, m, _ in play_edges])
g.es['hours'] = [hours for _, _, hours in play_edges]

In [ ]:
g.add_edges(purchase_edges)

In [ ]:
edge_type = ['PLAYED' for _ in play_edges] + ['PURCHASED' for _ in purchase_edges]
g.es['edge_type'] = edge_type

In [ ]:
user_id_ex = g.vs.select(steam_id_eq='151603712')[0].index
purchased_ex = g.es.select(_source_eq=user_id_ex, edge_type='PURCHASED')
print(len(list(purchased_ex)))

## Path based analytics in igraph

In [ ]:
paths = g.get_all_simple_paths(user_id_ex, cutoff=3, mode='all')
print(paths[:10])

In [ ]:
rec_game_ids = [path[3] for path in paths if len(path) == 4]

In [ ]:
game_names = [g.vs[game_id]['steam_id'] for game_id in rec_game_ids]

In [ ]:
neighbors = g.neighbors(user_id_ex)
purchased_games = [g.vs[node_id]['steam_id'] for node_id in g.neighbors(user_id_ex)]

In [ ]:
game_names = [game for game in game_names if game not in purchased_games]

In [ ]:
from collections import Counter
game_frequency = Counter(game_names)
print(game_frequency)